# FiLM-GraspNet A1 Training on Kaggle

Before running: choose **GPU T4 x2** and turn **Internet on** in Kaggle settings. Add a Kaggle Dataset containing `train_pilot.tar.gz`, `val_pilot.tar.gz`, `grconvnet3_pretrained.pth`, and `SHA256SUMS.txt`. This notebook uses only one T4. Replace the dataset name in Cell 3. The A1 code must be available in the GitHub repository before training.


# System Checking


In [2]:

import os
import sys
import torch
import subprocess
from pathlib import Path

print("=== KAGGLE ENVIRONMENT ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    device = torch.device("cuda:0")
    print("Selected device:", device)
else:
    raise RuntimeError("GPU is not enabled!")

print("\n=== NVIDIA GPU ===")
subprocess.run(["nvidia-smi"], check=True)

print("\n=== WORKSPACE ===")
print("Current directory:", os.getcwd())
print("/kaggle/input exists:", Path("/kaggle/input").exists())
print("/kaggle/working exists:", Path("/kaggle/working").exists())


=== KAGGLE ENVIRONMENT ===
Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA version: 13.0
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Selected device: cuda:0

=== NVIDIA GPU ===
Wed Nov 18 16:03:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             14W /   

# Git Clone


In [3]:

from pathlib import Path
import subprocess
import os

WORKSPACE = Path("/kaggle/working")
REPO_DIR = WORKSPACE / "film-graspnet"

GITHUB_URL = "https://github.com/nammannam/film-graspnet.git"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", GITHUB_URL, str(REPO_DIR)],
        check=True
    )
else:
    print("Repository already exists. Reusing it.")

os.chdir(REPO_DIR)

print("\n=== REPOSITORY ===")
print("Working directory:", os.getcwd())

print("\n=== GIT VERSION ===")
subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    check=True
)

print("\n=== PROJECT FILES ===")
for path in [
    "src/filmgrasp/training/train.py",
    "src/filmgrasp/models/grconvnet3.py",
    "requirements.txt",
]:
    print(path, ":", (REPO_DIR / path).exists())


Cloning into '/kaggle/working/film-graspnet'...



=== REPOSITORY ===
Working directory: /kaggle/working/film-graspnet

=== GIT VERSION ===
bbd8c89

=== PROJECT FILES ===
src/filmgrasp/training/train.py : True
src/filmgrasp/models/grconvnet3.py : True
requirements.txt : True


# Dependencies installation


In [4]:

from pathlib import Path
from importlib import metadata, util

REPO_DIR = Path("/kaggle/working/film-graspnet")

print("=== PROJECT REQUIREMENTS ===")
requirements = REPO_DIR / "requirements.txt"
print(requirements.read_text())

print("\n=== INSTALLED PACKAGES ===")

packages = {
    "torch": "torch",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "scipy": "scipy",
    "scikit-image": "skimage",
    "opencv-python": "cv2",
    "Pillow": "PIL",
    "matplotlib": "matplotlib",
    "tensorboard": "tensorboard",
    "tqdm": "tqdm",
    "openai-clip": "clip",
}

for package, module in packages.items():
    installed = util.find_spec(module) is not None
    if installed:
        try:
            version = metadata.version(package)
        except metadata.PackageNotFoundError:
            version = "present (version unknown)"
    else:
        version = "NOT INSTALLED"

    print(f"{package:18s} : {version}")

print("\n=== SOURCE VERSION ===")
train_source = (
    REPO_DIR / "src/filmgrasp/training/train.py"
).read_text()

print(
    "Dropout CLI supported:",
    "--dropout-prob" in train_source
)


=== PROJECT REQUIREMENTS ===
absl-py==2.3.1
brotlicffi==1.0.9.2
certifi==2025.10.5
cffi==2.0.0
charset-normalizer==3.4.4
clip==1.0
contourpy==1.3.0
cycler==0.12.1
filelock==3.19.1
filmgrasp==0.1.0
fonttools==4.60.2
fsspec==2025.10.0
ftfy==6.3.1
grpcio==1.80.1
hf-xet==1.8.0
huggingface_hub==0.36.2
idna==3.11
imagecodecs==2024.9.22
imageio==2.37.0
importlib_metadata==8.7.0
importlib_resources==6.5.2
kiwisolver==1.4.7
lazy_loader==0.4
Markdown==3.9
MarkupSafe==3.0.4
matplotlib==3.9.4
mkl_fft==1.3.11
mkl_random==1.2.8
mkl-service==2.4.0
networkx==3.2.1
numpy==1.23.5
packaging==26.3
pillow==11.1.0
pip==26.0.1
protobuf==6.33.6
pycparser==2.23
pyparsing==3.3.3
PySocks==1.7.1
python-dateutil==3.0.0
PyWavelets==1.6.0
PyYAML==6.0.3
regex==2026.1.15
requests==2.32.5
scikit-image==0.21.0
scipy==1.13.1
setuptools==80.9.0
six==1.17.0
tensorboard==2.21.0
tensorboard-data-server==0.7.2
tifffile==2024.6.18
torch==1.12.1
torchaudio==0.12.1
torchvision==0.13.1
tqdm==4.70.1
typing_extensions==4.15.0
tzdat

In [5]:

import sys
import subprocess
import importlib.util
import torch

print("=== INSTALLING MISSING DEPENDENCIES ===")

for package in ["ftfy", "regex"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", package],
            check=True
        )
    else:
        print(f"{package}: already installed")

# Install official OpenAI CLIP without replacing Kaggle's PyTorch.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "git+https://github.com/openai/CLIP.git"
    ],
    check=True
)

print("\n=== VERIFY CLIP ===")

import clip

print("CLIP location:", clip.__file__)
print("Available CLIP models:", clip.available_models())

assert "ViT-B/32" in clip.available_models()

print("\n=== VERIFY CUDA ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

assert torch.cuda.is_available()

print("\nCELL 4: PASS")


=== INSTALLING MISSING DEPENDENCIES ===
regex: already installed
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-r7ijd7xu


  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-r7ijd7xu


  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369548 sha256=4dd84b6c93fd247f80b26e58290fb933bb853e2d6559cfe1b5318f68d00dc8ec
  Stored in directory: /tmp/pip-ephem-wheel-cache-bwe0nxxk/wheels/cb/a8/74/5f32d6cf0407457f0f62737b6da5c14eb86b9cac476fdf630d
Successfully built clip

=== VERIFY CLIP ===
CLIP location: /usr/local/lib/python3.13/dist-packages/clip/__init__.py
Available CLIP models: ['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']

=== VERIFY CUDA ===
PyTorch: 2.11.0+cu130
CUDA available

In [6]:

import os
import sys
import importlib
from pathlib import Path

REPO_DIR = Path("/kaggle/working/film-graspnet")
SRC_DIR = REPO_DIR / "src"

os.chdir(REPO_DIR)

# Make the src-layout package importable in this notebook.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("=== FILMGRASP IMPORT CHECK ===")

modules = [
    "filmgrasp",
    "filmgrasp.data.dataset",
    "filmgrasp.models.grconvnet3",
    "filmgrasp.training.train",
    "filmgrasp.evaluation.grasp_metrics",
]

for name in modules:
    try:
        module = importlib.import_module(name)
        print(f"PASS: {name}")
        print("      ", getattr(module, "__file__", "namespace package"))
    except Exception as exc:
        print(f"FAIL: {name}")
        print(f"      {type(exc).__name__}: {exc}")
        raise

print("\nCELL 5: PASS")


=== FILMGRASP IMPORT CHECK ===
PASS: filmgrasp
       /kaggle/working/film-graspnet/src/filmgrasp/__init__.py
PASS: filmgrasp.data.dataset
       /kaggle/working/film-graspnet/src/filmgrasp/data/dataset.py
PASS: filmgrasp.models.grconvnet3
       /kaggle/working/film-graspnet/src/filmgrasp/models/grconvnet3.py


I0000 00:00:1795018599.839148      50 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


PASS: filmgrasp.training.train
       /kaggle/working/film-graspnet/src/filmgrasp/training/train.py
PASS: filmgrasp.evaluation.grasp_metrics
       /kaggle/working/film-graspnet/src/filmgrasp/evaluation/grasp_metrics.py

CELL 5: PASS


In [ ]:
%pip install -q ftfy regex tqdm tensorboard scipy scikit-image
%pip install -q --no-deps git+https://github.com/openai/CLIP.git
%pip install -q --no-deps -e /kaggle/working/film-graspnet

import clip
import filmgrasp
print("CLIP and FiLM-GraspNet imports: PASSED")


# Dataset


In [7]:

from pathlib import Path
import hashlib

INPUT = Path("/kaggle/input")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def find_dataset(name):
    matches = [
        p.parent
        for p in INPUT.rglob("manifest.json")
        if p.parent.name == name
        and (p.parent / "images").is_dir()
        and (p.parent / "grasps").is_dir()
    ]
    assert len(matches) == 1, (name, matches)
    return matches[0]

TRAIN_ROOT = find_dataset("train_pilot")
VAL_ROOT = find_dataset("val_pilot")

weights = list(INPUT.rglob("grconvnet3_pretrained.pth"))
assert len(weights) == 1, weights
PRETRAINED_PATH = weights[0]

expected_hashes = {
    "train": "7038dc3929deae9a729ed75d854e241f70729f41d22fea2b67520374c96225ff",
    "val": "4f93a67ab3d636489f784ca6d9c42de0ef99f7de0766ac1890cb1870fb31f7cc",
    "pretrained": "2fdb9867125eb7cbe5f05b0ee1c83fad29f157e39575b69da10868faf71349a4",
}

files = {
    "train": TRAIN_ROOT / "manifest.json",
    "val": VAL_ROOT / "manifest.json",
    "pretrained": PRETRAINED_PATH,
}

print("=== DATASET INTEGRITY ===")

for name, path in files.items():
    actual = sha256(path)
    assert actual == expected_hashes[name], (
        f"{name} SHA-256 mismatch: {actual}"
    )
    print(f"{name}: SHA-256 MATCH")

print("\n=== DATASET LOADER ===")

from filmgrasp.data.dataset import GraspAnythingPPDataset

train = GraspAnythingPPDataset(TRAIN_ROOT)
val = GraspAnythingPPDataset(VAL_ROOT)

print("Train samples:", len(train))
print("Validation samples:", len(val))

assert len(train) == 381
assert len(val) == 76

print("\n=== PATHS ===")
print("TRAIN_ROOT:", TRAIN_ROOT)
print("VAL_ROOT:", VAL_ROOT)
print("PRETRAINED_PATH:", PRETRAINED_PATH)

print("\nCELL 6: PASS")


=== DATASET INTEGRITY ===
train: SHA-256 MATCH
val: SHA-256 MATCH
pretrained: SHA-256 MATCH

=== DATASET LOADER ===
Train samples: 381
Validation samples: 76

=== PATHS ===
TRAIN_ROOT: /kaggle/input/datasets/namng04/grasp-anything-subset/train_pilot/train_pilot
VAL_ROOT: /kaggle/input/datasets/namng04/grasp-anything-subset/val_pilot/val_pilot
PRETRAINED_PATH: /kaggle/input/datasets/namng04/grasp-anything-subset/grconvnet3_pretrained.pth

CELL 6: PASS


# Model forward smoke-test

In [8]:

import torch
import gc

from filmgrasp.training.train import build_model

DEVICE = torch.device("cuda:0")

print("=== BUILD FiLM-GraspNet ===")

model = build_model(PRETRAINED_PATH, DEVICE)
model.eval()

print("Model initialized successfully!")

print("\n=== DEVICE CHECK ===")
print("Backbone:", next(model.backbone.parameters()).device)
print("FiLM:", next(model.film.parameters()).device)
print("CLIP:", next(model.text_encoder.parameters()).device)

assert next(model.backbone.parameters()).device.type == "cuda"
assert next(model.film.parameters()).device.type == "cuda"
assert next(model.text_encoder.parameters()).device.type == "cpu"
assert all(not p.requires_grad for p in model.text_encoder.parameters())

print("\n=== SAMPLE FORWARD PASS ===")

sample = train[0]

image = sample["image"].unsqueeze(0).to(DEVICE)
image = image - image.mean(dim=(1, 2, 3), keepdim=True)

prompt = sample["prompt"]

print("Input image:", tuple(image.shape))
print("Instruction:", prompt)

with torch.no_grad():
    outputs = model(image, [prompt])

head_names = ["pos", "cos", "sin", "width"]

assert len(outputs) == 4

for name, output in zip(head_names, outputs):
    print(
        f"{name:6s}: shape={tuple(output.shape)}, "
        f"finite={bool(torch.isfinite(output).all())}"
    )
    assert torch.isfinite(output).all()
    assert output.shape == (1, 1, 416, 416)

print("\n=== GPU MEMORY ===")
print(
    "Allocated:",
    round(torch.cuda.memory_allocated(0) / 1024**2, 2),
    "MiB"
)

# Release GPU memory before later training.
del model, image, outputs
gc.collect()
torch.cuda.empty_cache()

print("\nCELL 7: PASS")


=== BUILD FiLM-GraspNet ===


100%|███████████████████████████████████████| 338M/338M [00:04<00:00, 75.6MiB/s]


Model initialized successfully!

=== DEVICE CHECK ===
Backbone: cuda:0
FiLM: cuda:0
CLIP: cpu

=== SAMPLE FORWARD PASS ===
Input image: (1, 3, 416, 416)
Instruction: Lift vase by its container.
pos   : shape=(1, 1, 416, 416), finite=True
cos   : shape=(1, 1, 416, 416), finite=True
sin   : shape=(1, 1, 416, 416), finite=True
width : shape=(1, 1, 416, 416), finite=True

=== GPU MEMORY ===
Allocated: 21.52 MiB

CELL 7: PASS


In [9]:

import gc
import torch
import torch.nn.functional as F

from filmgrasp.training.train import build_model

DEVICE = torch.device("cuda:0")

print("=== TRAINING BACKWARD SMOKE TEST ===")

model = build_model(PRETRAINED_PATH, DEVICE)
model.train()

sample = train[0]

image = sample["image"].unsqueeze(0).to(DEVICE)
image = image - image.mean(dim=(1, 2, 3), keepdim=True)

head_names = ("pos", "cos", "sin", "width")

model.zero_grad(set_to_none=True)

outputs = model(image, [sample["prompt"]])

losses = {}

for name, output in zip(head_names, outputs):
    target = sample["targets"][name].to(DEVICE)
    target = target.reshape_as(output)

    losses[name] = F.smooth_l1_loss(output, target)
    print(f"{name} loss: {losses[name].item():.6f}")

total_loss = sum(losses.values())

print("Total loss:", round(total_loss.item(), 6))

assert torch.isfinite(total_loss)

total_loss.backward()

def check_grads(module, name):
    grads = [
        p.grad for p in module.parameters()
        if p.requires_grad and p.grad is not None
    ]

    assert grads, f"No gradients for {name}"
    assert all(torch.isfinite(g).all() for g in grads)

    nonzero = sum(torch.count_nonzero(g).item() for g in grads)
    print(f"{name} gradient elements (nonzero):", nonzero)
    assert nonzero > 0

check_grads(model.backbone, "Backbone")
check_grads(model.film, "FiLM")

assert all(
    not p.requires_grad
    for p in model.text_encoder.parameters()
)

print("CLIP frozen: PASS")
print("Backward pass: PASS")

print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated(0) / 1024**2, 2),
    "MiB"
)

del model, image, outputs, sample, losses, total_loss
gc.collect()
torch.cuda.empty_cache()

print("\nCELL 8: PASS")


=== TRAINING BACKWARD SMOKE TEST ===
pos loss: 0.032978
cos loss: 0.011828
sin loss: 0.003149
width loss: 0.017490
Total loss: 0.065445
Backbone gradient elements (nonzero): 1898211
FiLM gradient elements (nonzero): 131328
CLIP frozen: PASS
Backward pass: PASS
Peak GPU memory: 346.28 MiB

CELL 8: PASS


# Training


In [10]:

from pathlib import Path

# ===============================
# EXPERIMENT CONFIGURATION
# ===============================

EXPERIMENT = "A1"  # B0, A1, A2

CONFIGS = {
    "B0": {
        "run_name": "filmgrasp_kaggle_b0_seed42_100e",
        "dropout_prob": 0.0,
        "loss": "smooth_l1",
    },
    "A1": {
        "run_name": "filmgrasp_kaggle_a1_dropout_p01_seed42_100e",
        "dropout_prob": 0.1,
        "loss": "smooth_l1",
    },
    "A2": {
        "run_name": "filmgrasp_kaggle_a2_loss_seed42_100e",
        "dropout_prob": 0.0,
        "loss": None,  # To be defined in M6.3b
    },
}

assert EXPERIMENT in CONFIGS
CONFIG = CONFIGS[EXPERIMENT]

EPOCHS = 100
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
SEED = 42
NUM_WORKERS = 0
DEVICE = "cuda"

OUTPUT_ROOT = Path("/kaggle/working/film-graspnet/runs")
RUN_DIR = OUTPUT_ROOT / CONFIG["run_name"]

print("=== TRAINING CONFIGURATION ===")
print("Experiment:", EXPERIMENT)
print("Run:", CONFIG["run_name"])
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Seed:", SEED)
print("Dropout probability:", CONFIG["dropout_prob"])
print("Loss:", CONFIG["loss"])
print("Train:", TRAIN_ROOT)
print("Validation:", VAL_ROOT)
print("Pretrained:", PRETRAINED_PATH)
print("Output:", RUN_DIR)

print("\n=== SOURCE COMPATIBILITY ===")

source = (REPO_DIR / "src/filmgrasp/training/train.py").read_text()
supports_dropout = "--dropout-prob" in source

print("Dropout CLI supported:", supports_dropout)

if EXPERIMENT == "A1" and not supports_dropout:
    print("BLOCKED: A1 source has not been synchronized.")

if EXPERIMENT == "A2":
    print("BLOCKED: A2 loss implementation is not ready.")

print("\nCELL 9: CONFIGURATION CREATED")


=== TRAINING CONFIGURATION ===
Experiment: A1
Run: filmgrasp_kaggle_a1_dropout_p01_seed42_100e
Epochs: 100
Batch size: 4
Learning rate: 0.0001
Seed: 42
Dropout probability: 0.1
Loss: smooth_l1
Train: /kaggle/input/datasets/namng04/grasp-anything-subset/train_pilot/train_pilot
Validation: /kaggle/input/datasets/namng04/grasp-anything-subset/val_pilot/val_pilot
Pretrained: /kaggle/input/datasets/namng04/grasp-anything-subset/grconvnet3_pretrained.pth
Output: /kaggle/working/film-graspnet/runs/filmgrasp_kaggle_a1_dropout_p01_seed42_100e

=== SOURCE COMPATIBILITY ===
Dropout CLI supported: False
BLOCKED: A1 source has not been synchronized.

CELL 9: CONFIGURATION CREATED


In [18]:

import subprocess
from pathlib import Path

REPO_DIR = Path("/kaggle/working/film-graspnet")

print("=== UPDATE SOURCE FROM GITHUB ===")

subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd=REPO_DIR,
    check=True
)

print("\n=== CURRENT SOURCE VERSION ===")
subprocess.run(
    ["git", "log", "-1", "--oneline"],
    cwd=REPO_DIR,
    check=True
)

print("\n=== CHECK TRAINING CLI ===")
train_source = (
    REPO_DIR / "src/filmgrasp/training/train.py"
).read_text()

assert "--dropout-prob" in train_source, (
    "A1 implementation is missing. "
    "Check that the changes were pushed to GitHub."
)
print()
print("Dropout CLI: SUPPORTED")
print("CELL 10: PASS")


=== UPDATE SOURCE FROM GITHUB ===
Already up to date.

=== CURRENT SOURCE VERSION ===
2f50fce Add A1-Dropout FiLM-GraspNet  model

=== CHECK TRAINING CLI ===

Dropout CLI: SUPPORTED
CELL 10: PASS


In [15]:
# %load_ext tensorboard

# TB_DIR = RUN_DIR / "tensorboard"
# TB_DIR.mkdir(parents=True, exist_ok=True)

# print("Experiment:", EXPERIMENT)
# print("TensorBoard directory:", TB_DIR)

# %tensorboard --logdir {str(TB_DIR)} --port 6006 --reload_interval 10

Experiment: A1
TensorBoard directory: /kaggle/working/film-graspnet/runs/filmgrasp_kaggle_a1_dropout_p01_seed42_100e/tensorboard


<IPython.core.display.Javascript object>

In [ ]:

import os
import sys
import subprocess
from pathlib import Path

print("=== FiLM-GraspNet TRAINING ===")

# Configuration from Cell 9
assert EXPERIMENT in ("B0", "A1", "A2")

print(f"Training model: {EXPERIMENT}")

if EXPERIMENT == "A2":
    raise NotImplementedError(
        "A2 loss implementation is not ready yet."
    )

REPO_DIR = Path("/kaggle/working/film-graspnet")
OUTPUT_ROOT = REPO_DIR / "runs"
RUN_DIR = OUTPUT_ROOT / CONFIG["run_name"]

# Prevent accidental overwriting of previous runs.
if RUN_DIR.exists():
    raise FileExistsError(
        f"Run directory already exists: {RUN_DIR}"
    )

# Use only the first Tesla T4.
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "src")
env["CUDA_VISIBLE_DEVICES"] = "0"
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable,
    "-m", "filmgrasp.training.train",

    "--train-root", str(TRAIN_ROOT),
    "--val-root", str(VAL_ROOT),
    "--pretrained", str(PRETRAINED_PATH),

    "--output-root", str(OUTPUT_ROOT),
    "--run-name", CONFIG["run_name"],

    "--device", "cuda",
    "--seed", str(SEED),
    "--batch-size", str(BATCH_SIZE),
    "--lr", str(LEARNING_RATE),
    "--epochs", str(EPOCHS),
    "--num-workers", str(NUM_WORKERS),

    "--dropout-prob", str(CONFIG["dropout_prob"]),
]

# Verify the updated CLI before launching training.
help_result = subprocess.run(
    [sys.executable, "-m", "filmgrasp.training.train", "--help"],
    cwd=REPO_DIR,
    env=env,
    capture_output=True,
    text=True,
    check=True,
)

# assert "--dropout-prob" in help_result.stdout, (
#     "Current training source does not support Dropout."
# )

print("Experiment:", EXPERIMENT)
print("Run name:", CONFIG["run_name"])
print("Dropout:", CONFIG["dropout_prob"])
print("Train:", TRAIN_ROOT)
print("Validation:", VAL_ROOT)
print("Output:", RUN_DIR)

print("\n=== TRAINING COMMAND ===")
print(" ".join(cmd), flush=True)

print("\n=== START TRAINING ===", flush=True)

subprocess.run(
    cmd,
    cwd=REPO_DIR,
    env=env,
    check=True
)

print("\n=== TRAINING COMPLETED ===")
print("Run directory:", RUN_DIR)


=== FiLM-GraspNet TRAINING ===
Training model: A1
Experiment: A1
Run name: filmgrasp_kaggle_a1_dropout_p01_seed42_100e
Dropout: 0.1
Train: /kaggle/input/datasets/namng04/grasp-anything-subset/train_pilot/train_pilot
Validation: /kaggle/input/datasets/namng04/grasp-anything-subset/val_pilot/val_pilot
Output: /kaggle/working/film-graspnet/runs/filmgrasp_kaggle_a1_dropout_p01_seed42_100e

=== TRAINING COMMAND ===
/usr/bin/python3 -m filmgrasp.training.train --train-root /kaggle/input/datasets/namng04/grasp-anything-subset/train_pilot/train_pilot --val-root /kaggle/input/datasets/namng04/grasp-anything-subset/val_pilot/val_pilot --pretrained /kaggle/input/datasets/namng04/grasp-anything-subset/grconvnet3_pretrained.pth --output-root /kaggle/working/film-graspnet/runs --run-name filmgrasp_kaggle_a1_dropout_p01_seed42_100e --device cuda --seed 42 --batch-size 4 --lr 0.0001 --epochs 100 --num-workers 0 --dropout-prob 0.1

=== START TRAINING ===


I0000 00:00:1795020872.920701     466 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-18 16:56:39,855 epoch=1 train=381 val=76 train_loss=0.039501 val_loss=0.033659 accuracy=5.26% detection=100.00% iou=0.0355
2026-11-18 16:57:39,170 epoch=2 train=381 val=76 train_loss=0.028482 val_loss=0.030966 accuracy=1.32% detection=92.11% iou=0.0257
2026-11-18 16:58:37,985 epoch=3 train=381 val=76 train_loss=0.027342 val_loss=0.030611 accuracy=1.32% detection=86.84% iou=0.0324
2026-11-18 16:59:36,852 epoch=4 train=381 val=76 train_loss=0.026926 val_loss=0.030305 accuracy=0.00% detection=84.21% iou=0.0160
2026-11-18 17:00:35,869 epoch=5 train=381 val=76 train_loss=0.026690 val_loss=0.030153 accuracy=0.00% detection=84.21% iou=0.0274
2026-11-18 17:01:34,568 epoch=6 train=381 val=76 train_loss=0.026544 val_loss=0.030057 accuracy=0.00% detection=78.95% iou=0.0277
2026-11-18 17:02:33,284 epoch=7 train=381 val=76 train_loss=0.026353 val_loss=0.029967 accuracy

In [ ]:
# %cd /kaggle/working/film-graspnet

# !python -m filmgrasp.training.train \
#   --train-root /kaggle/input/datasets/namng04/grasp-anything-subset/train_pilot/train_pilot \
#   --val-root /kaggle/input/datasets/namng04/grasp-anything-subset/val_pilot/val_pilot \
#   --epochs 100 \
#   --batch-size 4 \
#   --lr 1e-4 \
#   --device cuda \
#   --num-workers 0 \
#   --seed 42 \
#   --pretrained {pretrained} \
#   --dropout-prob 0.1 \
#   --output-root /kaggle/working/runs \
#   --run-name filmgrasp_kaggle_a1_dropout_p01_seed42_100e


# Results

The run writes `best.pt`, `last.pt`, `metrics.csv`, `config.json`, `data_summary.json`, and `train.log` under `/kaggle/working/runs/filmgrasp_kaggle_a1_dropout_p01_seed42_100e/`. Use **Save Version** with output files, or download the run before ending the session. For a later resume, attach the saved A1 run as Kaggle input, copy it back to `/kaggle/working/runs/`, and add `--resume .../checkpoints/last.pt` to the same training command. Never resume from B0. Compare A1 and B0 using the same post-training evaluator and validation manifest.


In [ ]:
for name in ("checkpoints/best.pt", "checkpoints/last.pt", "metrics.csv", "config.json", "data_summary.json", "train.log"):
    assert (RUN / name).is_file(), f"Missing: {RUN / name}"
print("TRAINING OUTPUT CHECK PASSED:", RUN)
